### 0.初回設定
----

#### 0.1.仮想環境構築

In [ ]:
venv 設定i

#### 0.2.ライブラリ構築

In [ ]:
pip install lxml pandas

### 1.定義取得
----

In [ ]:
pwd

In [ ]:
cd ..

In [ ]:
import pandas as pd

def export_to_excel(data, path, sheet='Sheet1'):
    # 辞書をデータフレームに変換
    df = pd.DataFrame([data] if isinstance(data, dict) else data)
    
    try:
        # ファイルがあれば「シート追加 / 同名なら上書き」
        with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as w:
            df.to_excel(w, sheet_name=sheet, index=False)
    except FileNotFoundError:
        # ファイルがなければ「新規作成」
        with pd.ExcelWriter(path, engine='openpyxl', mode='w') as w:
            df.to_excel(w, sheet_name=sheet, index=False)
    print(f"「{path}」の「{sheet}」シートへ出力完了")

In [ ]:
# ===========================================
# ★★ファイル名の指定
# ===========================================
tbf = r"＊＊＊.twbx"
export_file = ""


In [ ]:
pwd

In [ ]:
cd ..

In [ ]:
from twbpatch import TwbWorkbook
wb = TwbWorkbook.open(tbf)

# データソースの読込/書出
datasources = [
    {"データソース名称": ds.caption, "データソース種別": ds.source_type}
    for ds in wb.list_datasources()
]

In [ ]:
# 

datasources

In [ ]:
lists = []

for dashboard in wb.list_dashboards():
    worksheet_names = "\n".join([
        str(worksheet.caption or worksheet.name or "")
        for worksheet in dashboard.worksheets
    ])

    lists.append({
        "種別": "ダッシュボード",
        "ダッシュボード/シート名": dashboard.caption,
        "表示/非表示": "表示" if dashboard.visible else "非表示",
        "組込シート一覧": worksheet_names,
    })

for ws in wb.list_worksheets():
    lists.append({
        "種別": "ワークシート",
        "ダッシュボード/シート名": ws.caption,
        "表示/非表示": "表示" if ws.visible else "非表示",
        "組込シート一覧": "-",
    })

lists

In [ ]:
fields = []

for f in wb.list_dashboard_fields():
    if f.type in {'行', '形状', 'ツールチップ', '色', 'ラベル', '詳細', 'ペイン', '列'}:
        fields.append({
            # "ワークシートID": f.id,
            "ワークシート名": f.worksheet,
            "ワークシート種別": f.type,
            "フィールド尺度": f.role,
            "フィールド名": f.caption,
        })
for f in wb.list_dashboard_fields():
    if f.type == 'フィルタ' in {}:
        fields.append({
            # "ワークシートID": f.id,
            "ワークシート名": f.worksheet,
            "ワークシート種別": f.type,
            "フィールド尺度": f.role,
            "フィールド名": f.caption,
        })
fields

In [ ]:
filters = []
for c in wb.list_dashboard_filter_controls():
    filters.append({
        "ダッシュボード": c.dashboard,
        "ワークシート": c.worksheet,
        "フィールド": c.field,
        "表示形式": c.mode,
        "値候補": c.value_scope_label or "",
        "適用範囲": c.apply_scope_label or "",
        "選択方式": c.selection_type or "",
        "値": "\n".join(str(v) for v in c.values),
    })
filters

In [ ]:
print(set(f.type for f in wb.list_dashboard_fields()))

In [ ]:
reference_lines = []

for line in wb.list_reference_lines():
    reference_lines.append({
        "ワークシート": line.worksheet,
        "軸フィールド": line.axis_caption or line.axis_column or "",
        "値フィールド": line.value_caption or line.value_column or "",
        "集計/式": line.formula or "",
        "スコープ": line.scope or "",
        "ラベル": line.label_type or "",
        "ツールチップ": line.tooltip_type or "",
    })

reference_lines

In [ ]:
parameters = []

for p in wb.list_parameters():
    parameters.append({
        "パラメータID": p.id or "",
        "パラメータ名": p.caption or p.name,
        "データ型": p.datatype or "",
        "現在値": p.value_display or p.value or "",
        "許容値タイプ": p.domain_type or "",
        "許容値": "\n".join(
            str(v.get("alias") or v.get("value") or "")
            for v in p.allowable_values
        ),
    })

parameters

In [ ]:
fields = []

for ds in wb.list_datasources():
    for col in ds.columns:
        fields.append({
            "データソースID": ds.id,
            "データソース名": ds.caption,
            "フィールドID": col.id,
            "フィールド名": col.caption,
            "フィールド尺度": col.role,
            "データ型": col.datatype,
            "フィールド種別": "計算フィールド" if col.is_calculated else "通常フィールド",
            "計算式_": col.formula,
            "参照フィールド": col.referenced_columns,
            "フォルダ": col.folder,
        })
fields

In [ ]:
# フィールド名変更
col = wb.rename_field(
    datasource="歯科医院×医師マスタ (kasai_dental_master_split_with_menu_reviews)",
    field="歯科医院ID",
    caption="歯科医院ID_rename",
)

In [ ]:
# フィールドをフォルダへ移動
col = wb.move_field_to_folder(
    datasource="歯科医院×医師マスタ (kasai_dental_master_split_with_menu_reviews)",
    field="歯科医院ID",
    folder="売上系",
)

In [ ]:
# 計算フィールド作成
col = wb.create_calculated_field(
    datasource="歯科医院×医師マスタ (kasai_dental_master_split_with_menu_reviews)",
    caption="粗利率",
    formula="[歯科医院ID] + 'ssssss'",
    datatype="real",
    role="measure",
    folder="KPI",
)

In [ ]:
wb.save(
    r".\マイ Tableau リポジトリ\ワークブック\test2.twb",
    overwrite=True,
)